In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import json
spark = SparkSession. \
builder. \
appName("week5-assignment-2"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
json_data = [
    '{"emp_id": 101, "name": "Arun", "department": "Engineering", "skills": ["Python", "Spark", "SQL"], "projects": [{"project_name": "DataLake Migration", "hours": 120}, {"project_name": "ETL Revamp", "hours": 80}]}',
    '{"emp_id": 102, "name": "Divya", "department": "Analytics", "skills": ["PowerBI", "SQL"], "projects": [{"project_name": "Sales Dashboard", "hours": 60}]}',
    '{"emp_id": 103, "name": "Karthik", "department": "Engineering", "skills": [], "projects": null}',
]


In [5]:
records = [json.loads(rec) for rec in json_data]
df = spark.createDataFrame(records)

print("=== Original DataFrame ===")
df.printSchema()
df.show(truncate=False)

=== Original DataFrame ===
root
 |-- department: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- projects: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)

+-----------+------+-------+-----------------------------------------------------------------------------------------------+--------------------+
|department |emp_id|name   |projects                                                                                       |skills              |
+-----------+------+-------+-----------------------------------------------------------------------------------------------+--------------------+
|Engineering|101   |Arun   |[{project_name -> DataLake Migration, hours -> 120}, {project_name -> ETL Revamp, hours -> 80}]|[Python, Spark, SQL]|
|Analytics  |102 

In [6]:
df_skills = df.select(
    "emp_id",
    "name",
    explode(col("skills")).alias("skill")
)

print("=== explode(skills): one row per skill (null/empty arrays dropped) ===")
df_skills.show(truncate=False)

=== explode(skills): one row per skill (null/empty arrays dropped) ===
+------+-----+-------+
|emp_id|name |skill  |
+------+-----+-------+
|101   |Arun |Python |
|101   |Arun |Spark  |
|101   |Arun |SQL    |
|102   |Divya|PowerBI|
|102   |Divya|SQL    |
+------+-----+-------+



In [8]:
df_skills_outer = df.select(
    "emp_id",
    "name",
    explode_outer(col("skills")).alias("skill")
)
print("=== explode_outer(skills): rows with empty/null arrays kept ===")
df_skills_outer.show(truncate=False)


=== explode_outer(skills): rows with empty/null arrays kept ===
+------+-------+-------+
|emp_id|name   |skill  |
+------+-------+-------+
|101   |Arun   |Python |
|101   |Arun   |Spark  |
|101   |Arun   |SQL    |
|102   |Divya  |PowerBI|
|102   |Divya  |SQL    |
|103   |Karthik|null   |
+------+-------+-------+



In [9]:
df_skills_pos = df.select(
    "emp_id",
    "name",
    posexplode(col("skills")).alias("position", "skill")
)

print("=== posexplode(skills): element index + value ===")
df_skills_pos.show(truncate=False)


=== posexplode(skills): element index + value ===
+------+-----+--------+-------+
|emp_id|name |position|skill  |
+------+-----+--------+-------+
|101   |Arun |0       |Python |
|101   |Arun |1       |Spark  |
|101   |Arun |2       |SQL    |
|102   |Divya|0       |PowerBI|
|102   |Divya|1       |SQL    |
+------+-----+--------+-------+



In [11]:
df_projects = (
    df.select(
        "emp_id",
        "name",
        "department",
        explode_outer(col("projects")).alias("project")
    )
)

print("=== explode_outer(projects): array of structs flattened to columns ===")
df_projects.show(truncate=False)

=== explode_outer(projects): array of structs flattened to columns ===
+------+-------+-----------+--------------------------------------------------+
|emp_id|name   |department |project                                           |
+------+-------+-----------+--------------------------------------------------+
|101   |Arun   |Engineering|{project_name -> DataLake Migration, hours -> 120}|
|101   |Arun   |Engineering|{project_name -> ETL Revamp, hours -> 80}         |
|102   |Divya  |Analytics  |{project_name -> Sales Dashboard, hours -> 60}    |
|103   |Karthik|Engineering|null                                              |
+------+-------+-----------+--------------------------------------------------+



In [12]:
df_project1= 
    df_projects.select(
        "emp_id",
        "name",
        "department",
        col("project.project_name").alias("project_name"),
        col("project.hours").alias("hours"),
    )

SyntaxError: invalid syntax (<ipython-input-12-116deeec86e8>, line 1)

In [ ]:
print("=== explode_outer(projects): array of structs flattened to columns ===")
df_projects.show(truncate=False)